# City and County Explorer
Pick any of California's 482 cities or 57 counties and see taxes and spending per resident over time, then compare cities side by side. Fiscal years 2003-2024, from the State Controller's Office (SCO), in constant dollars.

**How to read it**
- **Taxes** are what the city or county itself levies (property, sales, hotel, utility users, business license, franchise and other taxes). **Spending** is everything it spends, including enterprise services such as water, power, airports and hospitals, which many cities run and others leave to special districts. High spending per person often reflects those services, not higher taxes or waste.
- **Per person** uses the population the government reports to the SCO. San Francisco is both a city and a county and appears here as a city. Population is the SCO's own estimate, not the Census count; two cities' populations are swapped in some SCO years (Mt. Shasta and Mountain View) and are corrected here (`population_repaired` in `data/processed/local_entities.csv`).
- **Incomplete picture:** residents also pay county, school, special-district and state taxes not shown for a city. Counties spend on behalf of everyone in them, including cities' residents.
- The SCO changed its reporting format in FY2016-17, so spending (but much less so taxes) can shift at that year.

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from ca_taxes.io import PROCESSED
from ca_taxes.deflate import add_views, base_fy, LAST_ACTUAL_FY
from ca_taxes.style import stacked_area_toggle, base_layout, PALETTE, SOURCE_NOTE
pio.renderers.default = "plotly_mimetype"
ent = pd.read_csv(PROCESSED / "local_entities.csv")
mac = pd.read_csv(PROCESSED / "macro.csv").set_index("fy")
ent["f"] = ent.fy.map(mac.cpi_factor)
ent["tax_pp"] = ent.tax_k * 1000 * ent.f / ent.population
ent["spend_pp"] = ent.spending_total_k * 1000 * ent.f / ent.population
SCO = "Source: California State Controller's Office, Cities and Counties Annual Reports. Real " + str(base_fy()) + " dollars."

def explorer(kind, default):
    d = ent[ent.kind == kind]
    last = d[d.fy == d.fy.max()].set_index("entity").population
    names = list(last.sort_values(ascending=False).index)  # largest first
    years = sorted(d.fy.unique())
    def series(n, col):
        e = d[d.entity == n].set_index("fy")[col].reindex(years)
        return [None if pd.isna(v) else int(round(v)) for v in e]
    sub = lambda n: f"{kind} (population in parentheses, largest first). {SCO}"
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=years, y=series(default, "tax_pp"), name="Taxes per person", mode="lines+markers", line=dict(color=PALETTE[0]), hovertemplate="$%{y:,.0f}"))
    fig.add_trace(go.Scatter(x=years, y=series(default, "spend_pp"), name="Spending per person", mode="lines+markers", line=dict(color=PALETTE[1]), hovertemplate="$%{y:,.0f}"))
    # buttons swap the two traces' data (one trace pair, not one pair per city) to keep the page small
    buttons = [dict(label=f"{n} ({int(last[n]):,})", method="update",
                    args=[{"y": [series(n, "tax_pp"), series(n, "spend_pp")]},
                          {"title.text": f"{n}: taxes and spending per person<br><sup>{sub(n)}</sup>"}])
               for n in names]
    fig.update_layout(updatemenus=[dict(type="dropdown", buttons=buttons, x=0, y=1.0, xanchor="left", yanchor="bottom",
                                        active=names.index(default), showactive=True)],
                      yaxis_title=f"{base_fy()} $ per person", yaxis_tickprefix="$", xaxis_title="Fiscal year (ending June 30)")
    base_layout(fig, f"{default}: taxes and spending per person", sub(default))
    fig.add_vline(x=2016.5, line_dash="dot", line_color="grey", annotation_text="SCO reporting format changes", annotation_position="bottom left")
    return fig
explorer("City", "Los Angeles").show()

In [2]:
explorer("County", "Los Angeles").show()

## Highest and lowest taxes per person
Cities with at least 25,000 residents, latest year (nominal dollars). The top is tourist and commercial centers (Beverly Hills, Palm Springs, Santa Monica, Monterey) and Bay Area cities, plus San Francisco, which is a combined city-county; part of the gap is visitors, workers and shoppers who pay tax but are not counted as residents. The bottom is mostly inland suburbs and lower-income cities in the Inland Empire and Central Valley.

In [3]:
FY = int(ent.fy.max())
c = ent[(ent.kind == "City") & (ent.fy == FY) & (ent.population >= 25000)].copy()
c["tax_nom"] = c.tax_k * 1000 / c.population
c["spend_nom"] = c.spending_total_k * 1000 / c.population
def bars(df, title):
    df = df.sort_values("tax_nom")
    fig = go.Figure(go.Bar(y=df.entity, x=df.tax_nom, orientation="h", marker_color=PALETTE[0],
                           customdata=df[["population", "spend_nom"]], hovertemplate="%{y}<br>$%{x:,.0f} taxes per person<br>$%{customdata[1]:,.0f} spending per person<br>population %{customdata[0]:,.0f}<extra></extra>"))
    fig.update_layout(xaxis_title="Taxes per person ($)", xaxis_tickprefix="$", showlegend=False, hovermode="closest")
    base_layout(fig, title, f"Cities with 25,000+ residents, FY{FY}, nominal dollars. Source: California State Controller's Office", height=640)
    fig.update_layout(margin=dict(l=130, r=20, t=90, b=50))
    return fig
bars(c.nlargest(20, "tax_nom"), "20 cities with the highest taxes per person").show()
bars(c.nsmallest(20, "tax_nom"), "20 cities with the lowest taxes per person").show()

In [4]:
fig = go.Figure(go.Scatter(x=c.tax_nom, y=c.spend_nom, mode="markers", text=c.entity, customdata=c.population,
                           marker=dict(size=(c.population ** 0.5 / 25).clip(5, 40), color=PALETTE[0], opacity=0.55, line=dict(width=0.5, color="white")),
                           hovertemplate="%{text}<br>$%{x:,.0f} taxes per person<br>$%{y:,.0f} spending per person<br>population %{customdata:,.0f}<extra></extra>"))
fig.update_layout(xaxis=dict(title="Taxes per person ($, log scale)", type="log", tickprefix="$"),
                  yaxis=dict(title="Spending per person ($, log scale)", type="log", tickprefix="$"), showlegend=False, hovermode="closest")
base_layout(fig, "Taxes vs spending per person, cities with 25,000+ residents", f"FY{FY}, nominal dollars; bubble size is population. Spending includes utilities and other enterprise services. Source: California State Controller's Office", height=620)
fig.show()